# Study 3 (clutter-vlm) with Qwen3-VL-8B on Kaggle

This runs the experiment from [zoyakhan51/clutter-vlm](https://github.com/zoyakhan51/clutter-vlm) unchanged, at a fixed commit, with the model `qwen3vl8b` (`Qwen/Qwen3-VL-8B-Instruct`, revision pinned in `models.py`).

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **on**, phone-verified account, secret **`HF_TOKEN`** set under Add-ons → Secrets (a read token of a Hugging Face account that can see the `atml-shapes-vqa` datasets).

**How to run**
1. **Trial:** keep `STAGE = "trial"`. Save Version → Save & Run All (about 20–30 min). Look at the log: the printed first answers should start with `circle:` / `apple:` / `box:` and the scorers should exit with code 0.
2. **Main run:** set `STAGE = "full"` and `PART = "all"`, then Save & Run All.
   Kaggle stops a session after 12 hours. The 4B model took about 5 hours on one GPU. The 8B model is split over two GPUs and is roughly twice as slow, so it may not finish in time. If it gets stopped, run it in two parts instead:
   - `PART = "1"`: answers for `shapes` and `hard`.
   - `PART = "2"`: answers for `objects` and `lines`, nameability, then the analysis. Before running, attach the **output of the part 1 version** as input (Add Input → Your Work / Notebook output). Its CSVs are copied in automatically.
3. **Results:** `/kaggle/working/preds` (answers, scores, nameability, `environment_qwen3vl8b.json`) and `/kaggle/working/results` (`results_qwen3vl8b.json`, `summary_qwen3vl8b.txt`). Download them or share the notebook with the study owner.

In [ ]:
# ---- settings ----
STAGE = "trial"   # "trial" (20 images per version) or "full"
PART = "all"      # "all", "1" (shapes+hard) or "2" (objects+lines, nameability, analysis)
MODEL = "qwen3vl8b"
COMMIT = "cadcd07e62cb4d7e9adcb5e859fa973e6663e4f1"  # zoyakhan51/clutter-vlm, the commit the study 3b models run
LIMIT = "--per-version 20" if STAGE == "trial" else ""
assert STAGE in ("trial", "full") and PART in ("all", "1", "2")
print(STAGE, PART, MODEL, COMMIT[:8])

In [ ]:
!pip install -q transformers==5.22.1
!rm -rf /tmp/code && git clone -q https://github.com/zoyakhan51/clutter-vlm /tmp/code && cd /tmp/code && git checkout -q {COMMIT} && git log -1 --oneline
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# answers on the four datasets, each scored with its official vqa_scoring.py
DATASETS = {"all": "shapes,hard,objects,lines", "1": "shapes,hard", "2": "objects,lines"}[PART]
!cd /tmp/code && python run_predictions.py --model {MODEL} --split all --datasets {DATASETS} {LIMIT}

In [ ]:
# nameability of every squiggle and other_shapes shape (H2, Stage 0 naming check)
if PART in ("all", "2"):
    !cd /tmp/code && python run_nameability.py --model {MODEL} {LIMIT}

In [ ]:
# for part 2, bring in the part 1 answers from the attached input so the analysis sees all four datasets
import shutil
from pathlib import Path
preds = Path("/kaggle/working/preds")
if PART == "2":
    for name in ("shapes", "hard"):
        for pattern in (f"preds_{name}_{MODEL}.csv", f"scores_{name}_{MODEL}.csv"):
            found = sorted(Path("/kaggle/input").rglob(pattern))
            if found:
                shutil.copy(found[0], preds / pattern)
                print("copied", found[0])
            else:
                print("NOT FOUND in /kaggle/input:", pattern, "- attach the part 1 output")

In [ ]:
# H1-H8 with 95% bootstrap intervals, as fixed in docs/study3_methodology.pdf
if PART in ("all", "2"):
    !cd /tmp/code && python analysis.py --model {MODEL}
!ls -la /kaggle/working/preds /kaggle/working/results 2>/dev/null